# AENIN Baseline Comparison Notebook

Runs the 8 baseline models (GroupINN, BrainGNN, EV-GCN, AL-NEGAT, Ex-NEGAT, DeepASD, GNN-LSTM, MCDGLN) plus your AENIN model on the SAME real dataset pipeline (AAL atlas -> PLV graphs -> 18-d node features), for 5-fold CV and site-wise CV.

Sections:
1. Imports & setup
2. Feature pipeline (data loading, PLV graph construction, node features)
3. Baseline model architectures
4. Dataset bridge (PyG -> dense tensors)
5. Training / evaluation utilities
6. Run 5-fold CV
7. Run site-wise CV

## 1. Feature Pipeline

Mirrors your AENIN_lasso.ipynb data construction: atlas loading, ROI time-series extraction, PLV graph construction, 18-dim node features, subject scanning, dataset caching.

In [ ]:
"""
feature_pipeline.py

This is the REAL data pipeline, lifted from AENIN_lasso.ipynb, so the
8 baseline models and AENIN are trained/evaluated on the exact same
subjects, graphs, and features.

Pipeline:
    .nii / .nii.gz  -->  AAL ROI time series  -->  PLV matrix
        -->  adaptive threshold (A, A_w)  -->  18-dim node features
        -->  concat with N x N correlation matrix  -->  PyG Data object

Expected folder structure (same as your notebook):
    data_root/
        ASD/  sub-xxxx_..._.nii.gz   (label = 1)
        TD/   sub-xxxx_..._.nii.gz   (label = 0)

Site id is parsed from the filename prefix before the first "_"
(same convention as your notebook: site = Path(nii_path).name.split("_")[0]).

Usage:
    from feature_pipeline import load_or_build_dataset, Config

    cfg = Config()
    cfg.DATA_ROOT = "/path/to/Dataset_ABIDE_sorted"
    dataset = load_or_build_dataset(cfg, cache_path="dataset_cor.pt")
    # dataset: list[torch_geometric.data.Data], each with
    #   .x            (N, 18 + N)   node features (18 topo/signal + N-dim corr row)
    #   .edge_index   (2, E)
    #   .edge_attr    (E, 1)        PLV weight (incl. self loops)
    #   .y            (1,)          0 = TD, 1 = ASD
    #   .plv_matrix   (N, N)        full (unthresholded) PLV matrix
    #   .time_series  (T, N)        raw ROI BOLD signal  (added for dynamic baselines)
    #   .site         str           acquisition site id parsed from filename
"""

import logging
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import torch
from scipy.signal import hilbert
from scipy.sparse.csgraph import connected_components, shortest_path
from scipy.sparse import csr_matrix

import nibabel as nib
from nilearn import datasets
from nilearn.input_data import NiftiLabelsMasker

from torch_geometric.data import Data

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s",
                     datefmt="%H:%M:%S")
log = logging.getLogger("AENIN-data")


# --------------------------------------------------------------------------- #
# Config (subset relevant to data building; mirrors your notebook's Config)
# --------------------------------------------------------------------------- #

class Config:
    DATA_ROOT: str = "./data"
    ATLAS: str = "aal"
    N_ROIS: int = 116
    APS_ALPHA: float = 0.5
    MIN_EDGES: int = 30
    WAVELET_SCALES: list = [0.5, 1.0, 2.0]
    SEED: int = 42


def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --------------------------------------------------------------------------- #
# 1. Atlas loading
# --------------------------------------------------------------------------- #

def load_atlas(atlas_name: str):
    log.info(f"Loading atlas: {atlas_name.upper()}")
    if atlas_name.lower() == "aal":
        atlas = datasets.fetch_atlas_aal()
        atlas_img = nib.load(atlas.maps) if isinstance(atlas.maps, str) else atlas.maps
        roi_labels = list(atlas.labels)
        n_rois = 116
    elif atlas_name.lower() in ("ho", "harvard-oxford"):
        atlas = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
        atlas_img = nib.load(atlas.maps) if isinstance(atlas.maps, str) else atlas.maps
        roi_labels = list(atlas.labels)
        n_rois = len(roi_labels)
    else:
        raise ValueError(f"Unknown atlas: {atlas_name}. Use 'aal' or 'ho'.")

    atlas_data = atlas_img.get_fdata()
    atlas_values = sorted([int(v) for v in np.unique(atlas_data.astype(np.int32)) if int(v) > 0])
    atlas_values = atlas_values[:n_rois]
    value_to_pos = {v: i for i, v in enumerate(atlas_values)}
    if len(roi_labels) > n_rois:
        roi_labels = roi_labels[:n_rois]
    log.info(f"  Atlas loaded: {n_rois} ROIs")
    return atlas_img, n_rois, roi_labels, value_to_pos


# --------------------------------------------------------------------------- #
# 2. ROI time-series extraction (fixed AAL ordering)
# --------------------------------------------------------------------------- #

def extract_roi_timeseries_fixed_aal(nii_path, atlas_img, t_r=2.0, n_rois=116):
    masker = NiftiLabelsMasker(
        labels_img=atlas_img, standardize=True, detrend=True,
        low_pass=0.1, high_pass=0.01, t_r=t_r, verbose=0,
    )
    ts_partial = masker.fit_transform(nii_path)
    T = ts_partial.shape[0]

    atlas_nii = nib.load(atlas_img) if isinstance(atlas_img, str) else atlas_img
    atlas_data = atlas_nii.get_fdata()
    atlas_values = sorted([int(v) for v in np.unique(atlas_data.astype(np.int32)) if int(v) > 0])
    atlas_values = atlas_values[:n_rois]
    value_to_pos = {v: i for i, v in enumerate(atlas_values)}

    ts_full = np.zeros((T, n_rois), dtype=np.float32)
    extracted_labels = getattr(masker, "labels_", None)
    if extracted_labels is not None:
        extracted_labels = [int(v) for v in extracted_labels if int(v) > 0]
        for k, atlas_value in enumerate(extracted_labels):
            if k >= ts_partial.shape[1]:
                break
            if atlas_value in value_to_pos:
                ts_full[:, value_to_pos[atlas_value]] = ts_partial[:, k]
    else:
        K = min(ts_partial.shape[1], n_rois)
        ts_full[:, :K] = ts_partial[:, :K]

    return np.nan_to_num(ts_full, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


# --------------------------------------------------------------------------- #
# 3. Adaptive Phase Synchronization (APS) graph construction
# --------------------------------------------------------------------------- #

def compute_plv_matrix(time_series: np.ndarray) -> np.ndarray:
    T, N = time_series.shape
    analytic = hilbert(time_series, axis=0)
    phases = np.angle(analytic)
    plv = np.zeros((N, N), dtype=np.float32)
    for i in range(N):
        diff = phases[:, i:i + 1] - phases
        plv[i, :] = np.abs(np.mean(np.exp(1j * diff), axis=0))
    np.fill_diagonal(plv, 0.0)
    return plv


def adaptive_threshold(plv_matrix: np.ndarray, alpha: float = 0.5):
    vals = plv_matrix[np.triu_indices_from(plv_matrix, k=1)]
    mu, sigma = vals.mean(), vals.std()
    theta = mu + alpha * sigma
    A_w = plv_matrix.copy()
    A_w[A_w < theta] = 0.0
    A = (A_w > 0).astype(np.float32)
    return A, A_w, float(theta)


def build_edge_index_and_attr(A: np.ndarray, A_w: np.ndarray):
    src, dst = np.where(A > 0)
    weights = A_w[src, dst]
    N = A.shape[0]
    self_src = np.arange(N)
    self_dst = np.arange(N)
    self_w = np.ones(N, dtype=np.float32)
    src = np.concatenate([src, self_src])
    dst = np.concatenate([dst, self_dst])
    weights = np.concatenate([weights, self_w])
    edge_index = torch.tensor(np.stack([src, dst], axis=0), dtype=torch.long)
    edge_attr = torch.tensor(weights[:, None], dtype=torch.float32)
    return edge_index, edge_attr


# --------------------------------------------------------------------------- #
# 4. 18-dim node feature extraction (verbatim from your notebook)
# --------------------------------------------------------------------------- #

def _personalized_pagerank(A, alpha=0.85, max_iter=100):
    N = A.shape[0]
    deg = A.sum(axis=1, keepdims=True)
    deg[deg == 0] = 1.0
    P = A / deg
    r = np.ones(N, dtype=np.float64) / N
    teleport = np.ones(N, dtype=np.float64) / N
    for _ in range(max_iter):
        r_new = alpha * P.T @ r + (1 - alpha) * teleport
        if np.linalg.norm(r_new - r, 1) < 1e-6:
            break
        r = r_new
    return r.astype(np.float32)


def _harmonic_centrality(A):
    N = A.shape[0]
    sp = shortest_path(csr_matrix(A), directed=False, unweighted=True)
    sp[sp == 0] = np.inf
    hc = (1.0 / sp)
    np.fill_diagonal(hc, 0.0)
    return hc.sum(axis=1).astype(np.float32) / (N - 1)


def _k_core_numbers(A):
    N = A.shape[0]
    adj = (A > 0).astype(int)
    core = np.zeros(N, dtype=np.float32)
    remaining = np.ones(N, dtype=bool)
    k = 1
    while remaining.any():
        changed = True
        while changed:
            changed = False
            for i in np.where(remaining)[0]:
                deg = adj[i][remaining].sum() - adj[i, i]
                if deg < k:
                    remaining[i] = False
                    core[i] = k - 1
                    changed = True
        k += 1
        if k > N:
            break
    core[remaining] = k - 1
    return core


def _wavelet_energy(ts, scales):
    T, N = ts.shape
    energies = np.zeros((N, len(scales)), dtype=np.float32)
    for s_idx, scale in enumerate(scales):
        step = max(1, int(scale * 2))
        downsampled = ts[::step, :]
        energies[:, s_idx] = np.mean(downsampled ** 2, axis=0)
    return energies


def _hub_score(A):
    eigvals, eigvecs = np.linalg.eig(A @ A.T)
    idx = np.argmax(np.real(eigvals))
    hub = np.abs(np.real(eigvecs[:, idx]))
    hub = hub / (hub.max() + 1e-8)
    return hub.astype(np.float32)


def _core_periphery_score(A):
    core = _k_core_numbers(A)
    return (core / (core.max() + 1e-8)).astype(np.float32)


def _participation_score(A):
    n_components, labels = connected_components(A.astype(np.int32), directed=False)
    N = A.shape[0]
    deg = A.sum(axis=1)
    P = np.zeros(N)
    for i in range(N):
        if deg[i] == 0:
            continue
        s = 0
        for c in range(n_components):
            nodes = np.where(labels == c)[0]
            k_im = A[i, nodes].sum()
            s += (k_im / deg[i]) ** 2
        P[i] = 1 - s
    return P.astype(np.float32)


def _flow_betweenness(A):
    sp = shortest_path(A, directed=False)
    N = A.shape[0]
    score = np.zeros(N)
    for i in range(N):
        score[i] = np.isfinite(sp[i]).sum()
    score /= score.max() + 1e-8
    return score.astype(np.float32)


def extract_node_features(time_series, A, A_w, cfg):
    """18-dim per-ROI feature vector. Same indices as your notebook."""
    present_mask = (time_series.std(axis=0) > 1e-8).astype(np.float32)
    T, N = time_series.shape
    feats = np.zeros((N, 18), dtype=np.float32)

    deg = A.sum(axis=1)
    feats[:, 0] = deg / (N - 1 + 1e-8)
    feats[:, 1] = A_w.sum(axis=1)
    feats[:, 2] = _personalized_pagerank(A)
    feats[:, 3] = _harmonic_centrality(A)
    k_core = _k_core_numbers(A)
    feats[:, 4] = k_core / (k_core.max() + 1e-8)

    A_2 = A @ A
    np.fill_diagonal(A_2, 0)
    feats[:, 5] = A_2.sum(axis=1) / (N * (N - 1) + 1e-8)

    avg_nb_deg = np.zeros(N, dtype=np.float32)
    for i in range(N):
        nb = np.where(A[i] > 0)[0]
        avg_nb_deg[i] = deg[nb].mean() if len(nb) > 0 else 0.0
    feats[:, 6] = avg_nb_deg / (N + 1e-8)

    feats[:, 7] = time_series.mean(axis=0)
    feats[:, 8] = time_series.std(axis=0)

    wav = _wavelet_energy(time_series, cfg.WAVELET_SCALES)
    feats[:, 9:12] = wav
    feats[:, 13] = present_mask

    plv_nb = np.zeros(N, dtype=np.float32)
    for i in range(N):
        nb = np.where(A[i] > 0)[0]
        plv_nb[i] = A_w[i, nb].mean() if len(nb) > 0 else 0.0
    feats[:, 12] = plv_nb

    feats[:, 14] = _hub_score(A)
    feats[:, 15] = _core_periphery_score(A)
    feats[:, 16] = _participation_score(A)
    feats[:, 17] = _flow_betweenness(A)

    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    return feats.astype(np.float32)


# --------------------------------------------------------------------------- #
# 5. Subject scanning + per-subject processing + dataset building
# --------------------------------------------------------------------------- #

def load_subjects(data_root: str):
    """Scan data_root/ASD and data_root/TD for .nii/.nii.gz. ASD=1, TD=0."""
    root = Path(data_root)
    subjects = []
    for label_name, label_val in [("ASD", 1), ("TD", 0)]:
        folder = root / label_name
        if not folder.exists():
            log.warning(f"  Folder not found: {folder}")
            continue
        nii_files = sorted(list(folder.glob("*.nii")) + list(folder.glob("*.nii.gz")))
        log.info(f"  {label_name}: {len(nii_files)} subjects found")
        for f in nii_files:
            subjects.append((str(f), label_val))
    if len(subjects) == 0:
        raise FileNotFoundError(f"No .nii/.nii.gz files found under {data_root}/ASD/ or {data_root}/TD/")
    return subjects


def process_subject(nii_path: str, label: int, atlas_img, cfg: Config,
                     keep_time_series: bool = True) -> Data:
    """fMRI -> time series -> APS graph -> 18-d node feats -> PyG Data."""
    ts = extract_roi_timeseries_fixed_aal(nii_path, atlas_img, n_rois=cfg.N_ROIS)
    site = Path(nii_path).name.split("_")[0]

    plv = compute_plv_matrix(ts)
    corr = np.corrcoef(ts.T)
    corr = np.nan_to_num(corr)
    np.fill_diagonal(corr, 0.0)

    A, A_w, theta = adaptive_threshold(plv, alpha=cfg.APS_ALPHA)
    if A.sum() < cfg.MIN_EDGES:
        for alpha_relax in [0.3, 0.1, 0.0]:
            A, A_w, theta = adaptive_threshold(plv, alpha=alpha_relax)
            if A.sum() >= cfg.MIN_EDGES:
                break

    x18 = extract_node_features(ts, A, A_w, cfg)          # (N, 18)
    corr_feat = corr.astype(np.float32)                    # (N, N)
    x = np.concatenate([x18, corr_feat], axis=1).astype(np.float32)  # (N, 18+N)

    edge_index, edge_attr = build_edge_index_and_attr(A, A_w)

    n_edges = int(A.sum())
    density = n_edges / (x.shape[0] * (x.shape[0] - 1) + 1e-8)
    plv_full = plv.copy()
    np.fill_diagonal(plv_full, 0)

    data = Data(
        x=torch.tensor(x, dtype=torch.float32),
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=torch.tensor([label], dtype=torch.long),
        plv_matrix=torch.tensor(plv_full, dtype=torch.float32),
        n_edges=n_edges,
        density=density,
    )
    data.site = site
    if keep_time_series:
        # needed by the dynamic baselines (GNN-LSTM, MCDGLN); not in your
        # original cache, so re-build with keep_time_series=True if you
        # want those two baselines to use real sliding-window dynamics.
        data.time_series = torch.tensor(ts, dtype=torch.float32)
    return data


def build_dataset(subjects: list, atlas_img, cfg: Config, keep_time_series: bool = True) -> list:
    dataset = []
    n = len(subjects)
    failed = 0
    for idx, (nii_path, label) in enumerate(subjects):
        try:
            data = process_subject(nii_path, label, atlas_img, cfg, keep_time_series)
            dataset.append(data)
            if (idx + 1) % 10 == 0 or (idx + 1) == n:
                log.info(f"  Processed {idx + 1}/{n} subjects")
        except Exception as e:
            log.warning(f"  Failed [{Path(nii_path).name}]: {e}")
            failed += 1
    log.info(f"  Dataset built: {len(dataset)} subjects ({failed} failed)")
    return dataset


def load_or_build_dataset(cfg: Config, cache_path: Optional[str] = "dataset_cor.pt",
                           force_rebuild: bool = False, keep_time_series: bool = True):
    """
    Loads dataset.pt / dataset_cor.pt if it already exists (same as your
    `torch.load("dataset.pt", ...)` cells), otherwise builds it from
    cfg.DATA_ROOT and caches it.
    """
    if cache_path is not None and Path(cache_path).exists() and not force_rebuild:
        log.info(f"Loading cached dataset: {cache_path}")
        dataset = torch.load(cache_path, map_location="cpu", weights_only=False)
        log.info(f"  Loaded {len(dataset)} subjects")
        return dataset

    log.info(f"Building dataset from: {cfg.DATA_ROOT}")
    atlas_img, n_rois, roi_labels, value_to_pos = load_atlas(cfg.ATLAS)
    cfg.N_ROIS = n_rois
    subjects = load_subjects(cfg.DATA_ROOT)
    dataset = build_dataset(subjects, atlas_img, cfg, keep_time_series=keep_time_series)

    if cache_path is not None:
        torch.save(dataset, cache_path)
        log.info(f"  Cached dataset to: {cache_path}")
    return dataset


def balanced_subset(dataset, n_per_class: int = 400, seed: int = 42):
    """Matches your notebook's 'keep only N ASD + N TD' balancing step."""
    asd = [d for d in dataset if d.y.item() == 1]
    td = [d for d in dataset if d.y.item() == 0]
    rng = np.random.RandomState(seed)
    rng.shuffle(asd)
    rng.shuffle(td)
    out = asd[:n_per_class] + td[:n_per_class]
    rng.shuffle(out)
    return out


## 2. Baseline Model Architectures

All 8 baselines (+ registry), sharing the same dense (x, A) / (x, A, ts) input interface.

In [ ]:
"""
Baseline model implementations for comparison against AENIN.

All models share a common input interface so they can be dropped into the
same training / 5-fold and site-wise CV loops as AENIN:

    x : (B, N, F)   node feature matrix   (F = 18 in your pipeline)
    A : (B, N, N)   weighted adjacency / correlation matrix (PLV / Pearson r)
    ts: (B, T, N)   raw BOLD time series  (only used by the dynamic models:
                                            GNN-LSTM, MCDGLN)

Each model's forward() returns raw logits of shape (B, num_classes).

Models implemented (faithful-but-compact re-implementations, not the
authors' original repos, since most of these papers do not release code
that matches your exact feature set):

  1. GroupINN    - Yan et al. 2019
  2. BrainGNN    - Li et al. 2021
  3. EV-GCN      - Huang & Chung 2022
  4. AL-NEGAT    - Chen et al. 2024
  5. Ex-NEGAT    - Bhavna et al. 2025
  6. DeepASD     - Chen et al. 2024
  7. GNN-LSTM    - Dvornek et al. 2017
  8. MCDGLN      - Wang et al. 2025

Install deps:
    pip install torch --break-system-packages
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# --------------------------------------------------------------------------- #
# Shared building blocks
# --------------------------------------------------------------------------- #

class DenseGCNLayer(nn.Module):
    """Plain dense GCN layer: H' = act(D^-1/2 A D^-1/2 H W)."""

    def __init__(self, in_dim, out_dim, act=F.relu, bias=True):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=bias)
        self.act = act

    @staticmethod
    def normalize_adj(A, eps=1e-6):
        # A: (B, N, N), make symmetric + self loops, then D^-1/2 A D^-1/2
        A = A.clone()
        B, N, _ = A.shape
        I = torch.eye(N, device=A.device).unsqueeze(0).expand(B, N, N)
        A = A + I
        deg = A.sum(-1).clamp(min=eps)
        d_inv_sqrt = deg.pow(-0.5)
        D = torch.diag_embed(d_inv_sqrt)
        return D @ A @ D

    def forward(self, x, A_norm):
        h = A_norm @ x
        h = self.lin(h)
        return self.act(h) if self.act is not None else h


class DenseGATLayer(nn.Module):
    """Multi-head dense graph attention layer (GAT), edge-masked by A."""

    def __init__(self, in_dim, out_dim, heads=4, concat=True, dropout=0.1):
        super().__init__()
        self.heads = heads
        self.out_dim = out_dim
        self.concat = concat
        self.W = nn.Linear(in_dim, heads * out_dim, bias=False)
        self.a_src = nn.Parameter(torch.empty(heads, out_dim))
        self.a_dst = nn.Parameter(torch.empty(heads, out_dim))
        nn.init.xavier_uniform_(self.a_src.unsqueeze(0))
        nn.init.xavier_uniform_(self.a_dst.unsqueeze(0))
        self.dropout = nn.Dropout(dropout)
        self.leaky = nn.LeakyReLU(0.2)

    def forward(self, x, A_mask, edge_feat=None):
        # x: (B, N, Fin), A_mask: (B, N, N) binary / weighted mask
        B, N, _ = x.shape
        h = self.W(x).view(B, N, self.heads, self.out_dim)          # (B,N,H,D)
        src = (h * self.a_src).sum(-1)                              # (B,N,H)
        dst = (h * self.a_dst).sum(-1)                              # (B,N,H)
        e = self.leaky(src.unsqueeze(2) + dst.unsqueeze(1))         # (B,N,N,H)
        if edge_feat is not None:
            e = e + edge_feat.unsqueeze(-1)                         # optional edge bias
        mask = (A_mask.unsqueeze(-1) > 0)
        e = e.masked_fill(~mask, float('-1e9'))
        alpha = torch.softmax(e, dim=2)                             # over neighbours
        alpha = self.dropout(alpha)
        h = h.permute(0, 2, 1, 3)                                   # (B,H,N,D)
        alpha = alpha.permute(0, 3, 1, 2)                           # (B,H,N,N)
        out = alpha @ h                                             # (B,H,N,D)
        out = out.permute(0, 2, 1, 3)                                # (B,N,H,D)
        if self.concat:
            out = out.reshape(B, N, self.heads * self.out_dim)
        else:
            out = out.mean(dim=2)
        return out, alpha  # return attention for explainability models


def readout(x, mode="mean"):
    """Graph-level readout over node dimension. x: (B, N, F)."""
    if mode == "mean":
        return x.mean(dim=1)
    elif mode == "max":
        return x.max(dim=1).values
    elif mode == "meanmax":
        return torch.cat([x.mean(dim=1), x.max(dim=1).values], dim=-1)
    raise ValueError(mode)


def topk_pool(x, A, score, k_ratio=0.5):
    """Simple top-k node pooling (used by BrainGNN's R-pool)."""
    B, N, F_ = x.shape
    k = max(1, int(N * k_ratio))
    topk_idx = score.topk(k, dim=1).indices                         # (B,k)
    gate = torch.sigmoid(score.gather(1, topk_idx)).unsqueeze(-1)   # (B,k,1)
    x_idx = topk_idx.unsqueeze(-1).expand(-1, -1, F_)
    x_pool = x.gather(1, x_idx) * gate
    a_idx_row = topk_idx.unsqueeze(-1).expand(-1, -1, N)
    A_pool = A.gather(1, a_idx_row)
    a_idx_col = topk_idx.unsqueeze(1).expand(-1, k, -1)
    A_pool = A_pool.gather(2, a_idx_col)
    return x_pool, A_pool


# --------------------------------------------------------------------------- #
# 1. GroupINN  (Yan et al., 2019)
# --------------------------------------------------------------------------- #

class GroupINN(nn.Module):
    """
    Group-wise feature reduction GCN: learns a soft clustering / grouping
    matrix S that compresses N ROIs into K "functional groups", then runs
    graph convolutions on the reduced (K x K) graph.
    """

    def __init__(self, n_nodes, in_dim=18, n_groups=16, hidden=32, num_classes=2):
        super().__init__()
        self.n_groups = n_groups
        self.assign = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_groups)
        )
        self.gcn1 = DenseGCNLayer(in_dim, hidden)
        self.gcn2 = DenseGCNLayer(hidden, hidden)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, num_classes)
        )

    def forward(self, x, A, ts=None):
        S = torch.softmax(self.assign(x), dim=-1)          # (B, N, K) soft grouping
        A_norm = DenseGCNLayer.normalize_adj(A)
        A_group = S.transpose(1, 2) @ A_norm @ S            # (B, K, K) reduced graph
        x_group = S.transpose(1, 2) @ x                     # (B, K, F) reduced features
        A_group_norm = DenseGCNLayer.normalize_adj(A_group)
        h = self.gcn1(x_group, A_group_norm)
        h = self.gcn2(h, A_group_norm)
        g = readout(h, "mean")
        return self.classifier(g)


# --------------------------------------------------------------------------- #
# 2. BrainGNN  (Li et al., 2021)
# --------------------------------------------------------------------------- #

class ROIAwareConv(nn.Module):
    """Ra-GConv: a separate filter bank conditioned on ROI community id."""

    def __init__(self, in_dim, out_dim, n_communities=8):
        super().__init__()
        self.n_communities = n_communities
        self.community_embed = nn.Embedding(n_communities, in_dim)
        self.W = nn.Linear(in_dim, out_dim)

    def forward(self, x, A_norm, community_ids):
        # community_ids: (N,) long tensor, shared across batch
        comm = self.community_embed(community_ids)          # (N, Fin)
        x_mod = x * torch.sigmoid(comm).unsqueeze(0)         # ROI-aware reweight
        h = A_norm @ x_mod
        return F.relu(self.W(h))


class BrainGNN(nn.Module):
    def __init__(self, n_nodes, in_dim=18, hidden=32, n_communities=8,
                 pool_ratio=0.5, num_classes=2):
        super().__init__()
        self.register_buffer(
            "community_ids",
            torch.randint(0, n_communities, (n_nodes,))   # replace with real
        )                                                  # atlas-based community
        self.conv1 = ROIAwareConv(in_dim, hidden, n_communities)
        self.score1 = nn.Linear(hidden, 1)
        self.conv2 = ROIAwareConv(hidden, hidden, n_communities)
        self.pool_ratio = pool_ratio
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, num_classes)
        )

    def forward(self, x, A, ts=None):
        A_norm = DenseGCNLayer.normalize_adj(A)
        h1 = self.conv1(x, A_norm, self.community_ids)
        s1 = self.score1(h1).squeeze(-1)
        h1_pool, A_pool = topk_pool(h1, A, s1, self.pool_ratio)
        N_pool = h1_pool.shape[1]
        comm_pool = self.community_ids[:N_pool]              # approx after pooling
        A_pool_norm = DenseGCNLayer.normalize_adj(A_pool)
        h2 = self.conv2(h1_pool, A_pool_norm, comm_pool)
        g = readout(h2, "meanmax")
        return self.classifier(g)


# --------------------------------------------------------------------------- #
# 3. EV-GCN  (Huang & Chung, 2022)
# --------------------------------------------------------------------------- #

class EdgeWeightNet(nn.Module):
    """Learns a scalar edge weight from a pair of node features (MLP on |xi-xj|, xi+xj)."""

    def __init__(self, in_dim, hidden=32):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim * 2, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, x):
        B, N, Fd = x.shape
        xi = x.unsqueeze(2).expand(B, N, N, Fd)
        xj = x.unsqueeze(1).expand(B, N, N, Fd)
        pair = torch.cat([torch.abs(xi - xj), xi + xj], dim=-1)
        w = torch.sigmoid(self.mlp(pair)).squeeze(-1)        # (B, N, N)
        return w


class EVGCN(nn.Module):
    """
    Edge-Variational GCN: combines the empirical correlation matrix with a
    learned, feature-driven edge-weight matrix (variational edge estimation),
    then performs standard graph convolution on the fused adjacency.
    """

    def __init__(self, n_nodes, in_dim=18, hidden=32, num_classes=2):
        super().__init__()
        self.edge_net = EdgeWeightNet(in_dim, hidden=32)
        self.alpha = nn.Parameter(torch.tensor(0.5))          # fusion weight (learnable)
        self.gcn1 = DenseGCNLayer(in_dim, hidden)
        self.gcn2 = DenseGCNLayer(hidden, hidden)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, num_classes)
        )

    def forward(self, x, A, ts=None):
        w_learned = self.edge_net(x)
        a = torch.sigmoid(self.alpha)
        A_fused = a * A.abs() + (1 - a) * w_learned
        A_norm = DenseGCNLayer.normalize_adj(A_fused)
        h = self.gcn1(x, A_norm)
        h = self.gcn2(h, A_norm)
        g = readout(h, "mean")
        return self.classifier(g)


# --------------------------------------------------------------------------- #
# 4. AL-NEGAT  (Chen et al., 2024) - Attention-Learning Node-Edge GAT
# --------------------------------------------------------------------------- #

class AL_NEGAT(nn.Module):
    """
    Node-Edge GAT: attention is computed jointly from node features AND the
    edge weight (correlation strength) feeding an extra bias term into the
    attention logits, with multi-layer attention "learning" via stacked heads.
    """

    def __init__(self, n_nodes, in_dim=18, hidden=16, heads=4, num_classes=2):
        super().__init__()
        self.edge_proj = nn.Linear(1, heads)
        self.gat1 = DenseGATLayer(in_dim, hidden, heads=heads, concat=True)
        self.gat2 = DenseGATLayer(hidden * heads, hidden, heads=heads, concat=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden * heads, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, num_classes)
        )

    def forward(self, x, A, ts=None):
        edge_bias = self.edge_proj(A.unsqueeze(-1))            # (B,N,N,heads)
        h, _ = self.gat1(x, A, edge_feat=edge_bias.mean(-1, keepdim=False))
        h = F.elu(h)
        h, _ = self.gat2(h, A, edge_feat=edge_bias.mean(-1, keepdim=False))
        h = F.elu(h)
        g = readout(h, "mean")
        return self.classifier(g)


# --------------------------------------------------------------------------- #
# 5. Ex-NEGAT  (Bhavna et al., 2025) - Explainable Node-Edge GAT
# --------------------------------------------------------------------------- #

class ExNEGAT(nn.Module):
    """
    Same backbone as AL-NEGAT but exposes per-layer attention maps for
    post-hoc explainability (saliency over edges / ROIs), and adds an
    attention-entropy regularization term you can add to the training loss
    to encourage sparse, interpretable attention.
    """

    def __init__(self, n_nodes, in_dim=18, hidden=16, heads=4, num_classes=2):
        super().__init__()
        self.gat1 = DenseGATLayer(in_dim, hidden, heads=heads, concat=True)
        self.gat2 = DenseGATLayer(hidden * heads, hidden, heads=heads, concat=False)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, num_classes)
        )
        self.last_attn = None  # stash for explainability plots

    def forward(self, x, A, ts=None):
        h, alpha1 = self.gat1(x, A)
        h = F.elu(h)
        h, alpha2 = self.gat2(h, A)
        h = F.elu(h)
        self.last_attn = (alpha1.detach(), alpha2.detach())
        g = readout(h, "mean")
        return self.classifier(g)

    @staticmethod
    def attention_entropy_loss(alpha, eps=1e-9):
        # alpha: (B, H, N, N) softmax weights -> encourage low entropy (sparse, interpretable)
        p = alpha.clamp(min=eps)
        ent = -(p * p.log()).sum(-1).mean()
        return ent


# --------------------------------------------------------------------------- #
# 6. DeepASD  (Chen et al., 2024) - CNN over the connectivity matrix
# --------------------------------------------------------------------------- #

class DeepASD(nn.Module):
    """
    Treats the N x N correlation matrix as a single-channel image and the
    node feature matrix as an auxiliary channel-stack, passed through a
    2D-CNN ("connectome-CNN") followed by an MLP classifier.
    """

    def __init__(self, n_nodes, in_dim=18, num_classes=2):
        super().__init__()
        self.n_nodes = n_nodes
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2), nn.BatchNorm2d(16), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4)
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(in_dim, 32), nn.ReLU(), nn.Linear(32, 32)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 4 * 4 + 32, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x, A, ts=None):
        img = A.unsqueeze(1)                                   # (B,1,N,N)
        c = self.conv(img).flatten(1)                          # (B, 64*4*4)
        node_g = self.node_mlp(x).mean(dim=1)                  # (B,32)
        out = torch.cat([c, node_g], dim=-1)
        return self.classifier(out)


# --------------------------------------------------------------------------- #
# 7. GNN-LSTM  (Dvornek et al., 2017)
# --------------------------------------------------------------------------- #

class GNN_LSTM(nn.Module):
    """
    Sliding-window dynamic connectivity: for each window, build a correlation
    graph from the raw time series, run a GCN to get a graph embedding, then
    feed the sequence of window embeddings into an LSTM for temporal modeling.
    Requires `ts` (B, T, N) raw BOLD signals.
    """

    def __init__(self, n_nodes, in_dim=18, hidden=32, window=30, stride=10,
                 lstm_hidden=32, num_classes=2):
        super().__init__()
        self.window = window
        self.stride = stride
        self.gcn = DenseGCNLayer(in_dim, hidden)
        self.lstm = nn.LSTM(hidden, lstm_hidden, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, num_classes)
        )

    @staticmethod
    def _window_corr(ts_window, eps=1e-8):
        # ts_window: (B, w, N) -> (B, N, N) Pearson correlation
        x = ts_window - ts_window.mean(dim=1, keepdim=True)
        cov = torch.einsum('bwi,bwj->bij', x, x)
        std = x.std(dim=1, unbiased=False).clamp(min=eps)
        denom = std.unsqueeze(-1) * std.unsqueeze(-2) * ts_window.shape[1]
        corr = cov / denom.clamp(min=eps)
        return corr

    def forward(self, x, A, ts):
        # x is the static 18-d node features, reused at every time window
        B, T, N = ts.shape
        embeds = []
        for start in range(0, max(T - self.window, 1), self.stride):
            w_ts = ts[:, start:start + self.window, :]
            if w_ts.shape[1] < 2:
                continue
            A_t = self._window_corr(w_ts)
            A_t_norm = DenseGCNLayer.normalize_adj(A_t)
            h = self.gcn(x, A_t_norm)
            embeds.append(readout(h, "mean"))
        if len(embeds) == 0:                                   # fallback: single static graph
            A_norm = DenseGCNLayer.normalize_adj(A)
            embeds = [readout(self.gcn(x, A_norm), "mean")]
        seq = torch.stack(embeds, dim=1)                        # (B, W, hidden)
        out, (hN, cN) = self.lstm(seq)
        return self.classifier(hN[-1])


# --------------------------------------------------------------------------- #
# 8. MCDGLN  (Wang et al., 2025) - Multi-Channel Dynamic Graph Learning Net
# --------------------------------------------------------------------------- #

class MCDGLN(nn.Module):
    """
    Builds K dynamic graph "channels" from sliding windows of the BOLD time
    series (e.g. correlation, partial correlation-style precision proxy,
    and a thresholded binary channel), runs a separate GCN per channel per
    window, fuses channels with learned attention, then aggregates the
    temporal sequence with a GRU before classification.
    """

    def __init__(self, n_nodes, in_dim=18, hidden=24, window=30, stride=15,
                 n_channels=3, gru_hidden=32, num_classes=2):
        super().__init__()
        self.window = window
        self.stride = stride
        self.n_channels = n_channels
        self.channel_gcns = nn.ModuleList(
            [DenseGCNLayer(in_dim, hidden) for _ in range(n_channels)]
        )
        self.channel_attn = nn.Linear(hidden, 1)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(gru_hidden, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, num_classes)
        )

    @staticmethod
    def _make_channels(A_t, n_channels):
        chans = [A_t]
        if n_channels >= 2:
            chans.append((A_t.abs() > 0.3).float() * A_t)        # sparsified channel
        if n_channels >= 3:
            chans.append(torch.tanh(3.0 * A_t))                  # nonlinear-emphasis channel
        return chans[:n_channels]

    def forward(self, x, A, ts):
        B, T, N = ts.shape
        window_embeds = []
        for start in range(0, max(T - self.window, 1), self.stride):
            w_ts = ts[:, start:start + self.window, :]
            if w_ts.shape[1] < 2:
                continue
            A_t = GNN_LSTM._window_corr(w_ts)
            chans = self._make_channels(A_t, self.n_channels)
            chan_embeds = []
            for c_idx, A_c in enumerate(chans):
                A_norm = DenseGCNLayer.normalize_adj(A_c)
                h = self.channel_gcns[c_idx](x, A_norm)
                chan_embeds.append(readout(h, "mean"))
            chan_stack = torch.stack(chan_embeds, dim=1)          # (B, C, hidden)
            attn_w = torch.softmax(self.channel_attn(chan_stack), dim=1)
            fused = (chan_stack * attn_w).sum(dim=1)               # (B, hidden)
            window_embeds.append(fused)
        if len(window_embeds) == 0:
            A_norm = DenseGCNLayer.normalize_adj(A)
            window_embeds = [readout(self.channel_gcns[0](x, A_norm), "mean")]
        seq = torch.stack(window_embeds, dim=1)                    # (B, W, hidden)
        out, hN = self.gru(seq)
        return self.classifier(hN[-1])


# --------------------------------------------------------------------------- #
# Registry, for convenient lookup in the training script
# --------------------------------------------------------------------------- #

MODEL_REGISTRY = {
    "GroupINN": GroupINN,
    "BrainGNN": BrainGNN,
    "EV-GCN": EVGCN,
    "AL-NEGAT": AL_NEGAT,
    "Ex-NEGAT": ExNEGAT,
    "DeepASD": DeepASD,
    "GNN-LSTM": GNN_LSTM,
    "MCDGLN": MCDGLN,
}

# Models that require raw time series (dynamic graph construction)
DYNAMIC_MODELS = {"GNN-LSTM", "MCDGLN"}


## 3. Dataset Bridge

Wraps the real PyG `Data` objects from the feature pipeline into dense (x, A, ts) tensors for the baseline models.

In [ ]:
"""
Dataset wrapper bridging your REAL AENIN data pipeline (PyG `Data` objects
built in feature_pipeline.py / your notebook) to two consumption modes:

  1. Native PyG mode   -> use torch_geometric.loader.DataLoader directly on
                          the list of Data objects (this is what your AENIN
                          model's forward(data) expects).

  2. Dense mode        -> AENINDenseDataset + collate_fn below convert each
                          PyG Data into (x, A, ts) dense tensors so the 8
                          baseline models in models.py (GroupINN, BrainGNN,
                          EV-GCN, AL-NEGAT, Ex-NEGAT, DeepASD, GNN-LSTM,
                          MCDGLN) can train on the EXACT same subjects,
                          splits, and underlying graph/features as AENIN.

Both modes pull from the SAME cached dataset (dataset.pt / dataset_cor.pt),
so the comparison in train_compare.py is apples-to-apples.
"""

import numpy as np
import torch
from torch.utils.data import Dataset

# Config, load_or_build_dataset, balanced_subset already defined above in this notebook


# --------------------------------------------------------------------------- #
# PyG Data -> dense (x, A) conversion
# --------------------------------------------------------------------------- #

def pyg_to_dense_adj(data, n_rois: int) -> torch.Tensor:
    """
    Rebuild the dense (N, N) weighted adjacency actually used by AENIN's
    message passing, from edge_index/edge_attr — excluding the self-loops
    that build_edge_index_and_attr() adds (DenseGCNLayer adds its own).
    """
    A = torch.zeros((n_rois, n_rois), dtype=torch.float32)
    src, dst = data.edge_index
    w = data.edge_attr.squeeze(-1)
    keep = src != dst  # drop self loops; dense layers re-add them
    A[src[keep], dst[keep]] = w[keep]
    return A


def pyg_node_features_18(data, n_rois: int) -> torch.Tensor:
    """
    data.x is (N, 18 + N): your notebook concatenates the 18-dim topology/
    signal features with the N x N correlation matrix. Slice back the first
    18 columns for the baseline models (in_dim=18, matching models.py).
    """
    return data.x[:, :18].clone()


class AENINDenseDataset(Dataset):
    """
    Wraps a list of PyG `Data` objects (from feature_pipeline.load_or_build_dataset)
    and serves dense tensors for the 8 baseline models.
    """

    def __init__(self, pyg_dataset, n_rois: int = 116, max_T: int = None):
        self.data_list = pyg_dataset
        self.n_rois = n_rois
        self.max_T = max_T

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        d = self.data_list[idx]
        x = pyg_node_features_18(d, self.n_rois)             # (N, 18)
        A = pyg_to_dense_adj(d, self.n_rois)                  # (N, N) weighted

        if hasattr(d, "time_series") and d.time_series is not None:
            ts = d.time_series.clone()                        # (T, N)
        else:
            # Dynamic baselines (GNN-LSTM, MCDGLN) need raw BOLD signal.
            # Your cached dataset.pt may not include it (only AENIN-ready
            # x/edge_index/edge_attr/plv_matrix are saved) — rebuild with
            # feature_pipeline.build_dataset(..., keep_time_series=True)
            # if you need those two baselines. Fallback: zero placeholder.
            T_fallback = self.max_T or 1
            ts = torch.zeros((T_fallback, self.n_rois), dtype=torch.float32)

        if self.max_T is not None:
            T = ts.shape[0]
            if T >= self.max_T:
                ts = ts[:self.max_T]
            else:
                pad = torch.zeros((self.max_T - T, self.n_rois), dtype=torch.float32)
                ts = torch.cat([ts, pad], dim=0)

        return {
            "x": x,
            "A": A,
            "ts": ts,
            "y": d.y.view(-1)[0].long(),
            "site": getattr(d, "site", "UNKNOWN_SITE"),
        }


def collate_fn(batch):
    x = torch.stack([b["x"] for b in batch], dim=0)
    A = torch.stack([b["A"] for b in batch], dim=0)
    ts = torch.stack([b["ts"] for b in batch], dim=0)
    y = torch.stack([b["y"] for b in batch], dim=0)
    sites = [b["site"] for b in batch]
    return {"x": x, "A": A, "ts": ts, "y": y, "site": sites}


# --------------------------------------------------------------------------- #
# Convenience: site / label arrays for sklearn CV splitters
# --------------------------------------------------------------------------- #

def get_labels_and_sites(pyg_dataset):
    labels = np.array([d.y.item() for d in pyg_dataset])
    sites = np.array([getattr(d, "site", "UNKNOWN_SITE") for d in pyg_dataset])
    return labels, sites


## 4. Training / Evaluation + 5-Fold CV / Site-wise CV

Runs every baseline (and AENIN, once you register it in MODEL_REGISTRY) on identical splits and features, and prints a comparison summary table.

In [ ]:
"""
Run 5-fold CV and Site-wise CV for ALL baselines (+ your AENIN model) on
the SAME data, SAME splits, SAME features -> produces a fair, reviewer-proof
comparison table (accuracy, sensitivity, specificity, F1, AUC per model).

Usage:
    python train_compare.py --mode 5fold
    python train_compare.py --mode site

Plug in:
    1. `load_subjects()`      -> build your list-of-dict subjects (see dataset.py)
    2. `build_node_features`  -> your feature-extraction function (18-d feats)
    3. `AENIN`                -> import your own model class and register it
"""

import argparse
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
import subprocess

# MODEL_REGISTRY, DYNAMIC_MODELS already defined above in this notebook
# AENINDenseDataset, collate_fn, get_labels_and_sites already defined above in this notebook
# Config, load_or_build_dataset, balanced_subset already defined above in this notebook
DataConfig = Config

# >>> plug in your own AENIN model here (PyG-based; needs its own training
#     loop with torch_geometric.loader.DataLoader since it expects forward(data),
#     not forward(x, A) like the dense baselines). See note in run_5fold(). >>>
# from your_aenin_module import AENIN
# MODEL_REGISTRY["AENIN"] = AENIN
def pick_gpu_with_max_free_memory():
        if not torch.cuda.is_available():
            print("CUDA not available. Using CPU.")
            return torch.device("cpu")

        try:
            result = subprocess.check_output(
                [
                    "nvidia-smi",
                    "--query-gpu=index,memory.free,memory.total",
                    "--format=csv,noheader,nounits",
                ],
                encoding="utf-8"
            )

            best_gpu = None
            best_free = -1
            for line in result.strip().split("\n"):
                idx, free_mem, total_mem = [x.strip() for x in line.split(",")]
                idx = int(idx)
                free_mem = int(free_mem)
                total_mem = int(total_mem)
                print(f"GPU {idx}: free={free_mem} MB / total={total_mem} MB")
                if free_mem > best_free:
                    best_free = free_mem
                    best_gpu = idx

            if best_gpu is not None:
                print(f"Selected GPU {best_gpu}")
                return torch.device(f"cuda:{best_gpu}")
        except Exception as e:
            print("nvidia-smi query failed:", e)

        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")



    # # ── Reproducibility
    # SEED: int = 42



DEVICE = pick_gpu_with_max_free_memory()


def train_one_fold(model_name, n_nodes, in_dim, num_classes,
                    train_ds, val_ds, epochs=60, lr=1e-3, batch_size=8):
    ModelCls = MODEL_REGISTRY[model_name]
    model = ModelCls(n_nodes=n_nodes, in_dim=in_dim, num_classes=num_classes).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                             collate_fn=collate_fn)

    best_val_acc, best_state = 0.0, None
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            x, A, ts, y = (batch["x"].to(DEVICE), batch["A"].to(DEVICE),
                            batch["ts"].to(DEVICE), batch["y"].to(DEVICE))
            opt.zero_grad()
            logits = model(x, A, ts) if model_name in DYNAMIC_MODELS else model(x, A)
            loss = crit(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()

        # quick val check each epoch, keep best checkpoint
        val_acc, *_ = evaluate(model, model_name, val_loader)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, val_loader


@torch.no_grad()
def evaluate(model, model_name, loader):
    model.eval()
    all_y, all_pred, all_prob = [], [], []
    for batch in loader:
        x, A, ts, y = (batch["x"].to(DEVICE), batch["A"].to(DEVICE),
                        batch["ts"].to(DEVICE), batch["y"].to(DEVICE))
        logits = model(x, A, ts) if model_name in DYNAMIC_MODELS else model(x, A)
        prob = torch.softmax(logits, dim=-1)[:, 1]
        pred = logits.argmax(dim=-1)
        all_y.extend(y.cpu().numpy().tolist())
        all_pred.extend(pred.cpu().numpy().tolist())
        all_prob.extend(prob.cpu().numpy().tolist())

    acc = accuracy_score(all_y, all_pred)
    f1 = f1_score(all_y, all_pred, zero_division=0)
    sens = recall_score(all_y, all_pred, zero_division=0)               # recall = sensitivity
    spec = recall_score(all_y, all_pred, pos_label=0, zero_division=0)  # specificity
    try:
        auc = roc_auc_score(all_y, all_prob)
    except ValueError:
        auc = float("nan")
    return acc, f1, sens, spec, auc


def run_5fold(pyg_dataset, n_nodes, in_dim=18, num_classes=2, n_splits=5, epochs=60):
    labels, _sites = get_labels_and_sites(pyg_dataset)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    full_ds = AENINDenseDataset(pyg_dataset, n_rois=n_nodes)

    results = {name: [] for name in MODEL_REGISTRY}
    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        print(f"\n===== Fold {fold + 1}/{n_splits} "
              f"(train={len(tr_idx)}, val={len(va_idx)}) =====")
        train_ds = Subset(full_ds, tr_idx)
        val_ds = Subset(full_ds, va_idx)
        for name in MODEL_REGISTRY:
            model, val_loader = train_one_fold(name, n_nodes, in_dim, num_classes,
                                                train_ds, val_ds, epochs=epochs)
            acc, f1, sens, spec, auc = evaluate(model, name, val_loader)
            results[name].append((acc, f1, sens, spec, auc))
            print(f"  {name:10s} | acc={acc:.4f} f1={f1:.4f} "
                  f"sens={sens:.4f} spec={spec:.4f} auc={auc:.4f}")

    print("\n===== 5-fold CV summary (mean ± std) =====")
    for name, vals in results.items():
        arr = np.array(vals)
        means, stds = arr.mean(axis=0), arr.std(axis=0)
        print(f"{name:10s} | "
              f"acc={means[0]:.4f}±{stds[0]:.4f} "
              f"f1={means[1]:.4f}±{stds[1]:.4f} "
              f"sens={means[2]:.4f}±{stds[2]:.4f} "
              f"spec={means[3]:.4f}±{stds[3]:.4f} "
              f"auc={means[4]:.4f}±{stds[4]:.4f}")
    return results


def run_sitewise(pyg_dataset, n_nodes, in_dim=18, num_classes=2, epochs=60):
    labels, sites = get_labels_and_sites(pyg_dataset)
    logo = LeaveOneGroupOut()
    full_ds = AENINDenseDataset(pyg_dataset, n_rois=n_nodes)

    results = {name: [] for name in MODEL_REGISTRY}
    for fold, (tr_idx, va_idx) in enumerate(logo.split(np.zeros(len(labels)), labels, sites)):
        held_out_site = sites[va_idx[0]]
        print(f"\n===== Held-out site: {held_out_site} "
              f"(train={len(tr_idx)}, val={len(va_idx)}) =====")
        train_ds = Subset(full_ds, tr_idx)
        val_ds = Subset(full_ds, va_idx)
        for name in MODEL_REGISTRY:
            model, val_loader = train_one_fold(name, n_nodes, in_dim, num_classes,
                                                train_ds, val_ds, epochs=epochs)
            acc, f1, sens, spec, auc = evaluate(model, name, val_loader)
            results[name].append((held_out_site, acc, f1, sens, spec, auc))
            print(f"  {name:10s} | acc={acc:.4f} f1={f1:.4f} "
                  f"sens={sens:.4f} spec={spec:.4f} auc={auc:.4f}")

    print("\n===== Site-wise CV summary (mean ± std across sites) =====")
    for name, vals in results.items():
        arr = np.array([v[1:] for v in vals], dtype=np.float32)
        means, stds = arr.mean(axis=0), arr.std(axis=0)
        print(f"{name:10s} | "
              f"acc={means[0]:.4f}±{stds[0]:.4f} "
              f"f1={means[1]:.4f}±{stds[1]:.4f} "
              f"sens={means[2]:.4f}±{stds[2]:.4f} "
              f"spec={means[3]:.4f}±{stds[3]:.4f} "
              f"auc={means[4]:.4f}±{stds[4]:.4f}")
    return results


# ============================================================================
# 5. Configure and Run
# ============================================================================
# Edit these settings, then run the cells below.

DATA_ROOT    = "./data"  # folder with ASD/ and TD/ subfolders
CACHE_PATH   = "dataset_cor.pt"   # matches your original notebook dataset.pt / dataset_cor.pt
FORCE_REBUILD = False             # True -> rebuild from raw .nii files instead of using the cache
BALANCED_N   = 400                # subjects per class to keep (400+400=800); set 0 to use full dataset
EPOCHS       = 100
MODE         = "5fold"            # "5fold" or "site"

cfg = DataConfig()
cfg.DATA_ROOT = DATA_ROOT

# Loads dataset_cor.pt / dataset.pt if present, otherwise builds it from raw
# .nii files under DATA_ROOT and caches it.
pyg_dataset = load_or_build_dataset(cfg, cache_path=CACHE_PATH, force_rebuild=FORCE_REBUILD)

if BALANCED_N > 0:
    pyg_dataset = balanced_subset(pyg_dataset, n_per_class=BALANCED_N, seed=cfg.SEED)
    print(f"Using balanced subset: {len(pyg_dataset)} subjects ({BALANCED_N} ASD + {BALANCED_N} TD)")

n_nodes = pyg_dataset[0].x.shape[0]   # number of ROIs (116 for AAL)
print(f"N ROIs: {n_nodes}  |  Total subjects: {len(pyg_dataset)}")


In [ ]:
run_5fold(pyg_dataset, n_nodes, epochs=EPOCHS)

### Run 5-Fold Cross-Validation

In [ ]:
if MODE == "5fold":
    results_5fold = run_5fold(pyg_dataset, n_nodes, epochs=EPOCHS)


### Run Site-wise Cross-Validation

In [ ]:
if MODE == "site":
    results_site = run_sitewise(pyg_dataset, n_nodes, epochs=EPOCHS)
